In [0]:

source_path='/Volumes/cdc_catalog/cdc_demo/cdc_files/'

print(f'Source Path --> {source_path}')

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

bronze_stream=(spark.readStream
    .format('cloudFiles')
    .option('cloudFiles.format','json')
    .option('cloudFiles.schemaLocation', f'{source_path}_schema/')
    .option('cloudFIles.schemaEvolutionMode','addNewColumns')
    .load(source_path)
    )

bronze_stream.printSchema()

bronze_stream=bronze_stream.withColumn('customer_id',col('customer_id').cast('long'))\
    .withColumn('salary',col('salary').cast('decimal(12,2)'))\
    .withColumn('updated_at',col('updated_at').cast('timestamp'))


bronze_stream.writeStream.format('delta')\
    .option('checkpointLocation',f'{source_path}_checkpoint/bronze_customer_cdc')\
    .option('delta.enableChangeDataFeed',True)\
    .option('delta.enabledeletetionVectors',True)\
    .trigger(availableNow=True)\
    .toTable('cdc_catalog.bronze.bronze_customer_cdc')